# Meteo Data

Weather data is essential for energy data analysis — outdoor temperature drives heating demand, solar radiation affects PV production, and humidity influences cooling loads. In Switzerland, two main **open data sources** provide free weather station data:

| Provider | Description | Data Access | Resolution |
|---|---|---|---|
| **MeteoSwiss** | Federal Office of Meteorology (~160 stations) | Station lookup only | — |
| **Agrometeo** | Agricultural weather network (~217 stations) | Station lookup + data download | Hourly |

The `pyedautils` package provides a unified interface to both providers. This chapter shows how to:

1. Find the nearest weather station to any Swiss location
2. Download hourly weather data from Agrometeo
3. Compare both station networks

```{note}
Both providers require an internet connection. All API calls go to public endpoints — no API key needed.
```

In [ ]:
import pandas as pd

(meteo-station-lookup)=
## Station Lookup

Both providers let you find the nearest weather station to a given location. The typical workflow is:

1. Convert a **postal code (PLZ)** to GPS coordinates
2. Find the **nearest station** that has the sensor you need

### Helper: PLZ to Coordinates

The `geopy` module converts any Swiss postal code to latitude/longitude and altitude.

In [ ]:
from pyedautils.geopy import get_coordindates_ch_plz, get_altitude_lat_long

plz = 6048  # Horw
coord = get_coordindates_ch_plz(plz)
altitude = get_altitude_lat_long(coord[0], coord[1])

print(f"PLZ {plz}: lat={coord[0]:.4f}, lon={coord[1]:.4f}, altitude={altitude:.0f} m")

### MeteoSwiss

MeteoSwiss operates Switzerland's official automatic weather stations. The `find_nearest_station()` function returns the **station abbreviation** (e.g. `"LUZ"` for Luzern) of the closest station that has the requested sensor and is within ±150 m altitude.

In [ ]:
from pyedautils.meteo_swiss import find_nearest_station as find_meteoswiss

station_ms = find_meteoswiss(coord[0], coord[1], altitude, sensor="temp")
print(f"Nearest MeteoSwiss station for temperature: {station_ms}")

You can search for different sensor types: `"temp"`, `"globrad"` (global radiation), `"relhum"` (relative humidity), `"rain"`.

In [ ]:
for sensor in ["temp", "globrad", "relhum", "rain"]:
    station = find_meteoswiss(coord[0], coord[1], altitude, sensor=sensor)
    print(f"  {sensor:8s} → {station}")

### Agrometeo

[Agrometeo](https://www.agrometeo.ch) is a Swiss agricultural weather network with ~217 stations. Unlike MeteoSwiss, it also provides **direct data download** via a public API.

In [ ]:
from pyedautils.agroweather import find_nearest_station as find_agro

station_agro = find_agro(coord[0], coord[1], sensor="temp")
print(f"Nearest Agrometeo station for temperature: {station_agro}")

(meteo-station-overview)=
## Station Networks

Both providers expose a function to list all available stations. This is useful for understanding coverage and available sensors.

In [ ]:
from pyedautils.meteo_swiss import get_current_station_data

stations_ms = get_current_station_data()
print(f"MeteoSwiss: {len(stations_ms)} stations")
stations_ms[["Abk.", "Station", "Stationshöhe m ü. M."]].head()

In [ ]:
from pyedautils.agroweather import get_station_data

stations_agro = get_station_data()
print(f"Agrometeo: {len(stations_agro)} stations")
stations_agro[["id", "name", "lat", "lon", "altitude"]].head()

(meteo-download-data)=
## Downloading Weather Data

Agrometeo provides a public API for downloading **hourly weather data**. There are two ways to get data:

1. **By station ID** — if you already know the station
2. **By postal code (PLZ)** — automatically finds the nearest station(s)

### Download by Station ID

Use `download_data()` when you know the station ID (e.g. from `find_nearest_station()`).

In [ ]:
from pyedautils.agroweather import download_data

df_station = download_data(
    station_id=station_agro,
    start_date="2024-06-01",
    end_date="2024-06-07",
    sensors=["temp", "relhum"]
)

print(f"Shape: {df_station.shape}")
df_station.head()

### Download by Postal Code

The convenience function `download_data_by_plz()` handles the full workflow: PLZ → coordinates → nearest station(s) → download. If different sensors are closest to different stations, it automatically merges the results.

In [ ]:
from pyedautils.agroweather import download_data_by_plz

df_plz = download_data_by_plz(
    plz=6048,
    start_date="2024-06-01",
    end_date="2024-06-07",
    sensors=["temp", "globrad", "relhum", "rain"]
)

print(f"Shape: {df_plz.shape}")
print(f"Period: {df_plz.index.min()} to {df_plz.index.max()}")
df_plz.head()

In [ ]:
df_plz.describe()

### Quick Visualization

Let's plot the downloaded data to verify it looks reasonable.

In [ ]:
from pyedautils.plots import plot_timeseries

fig = plot_timeseries(
    df_plz,
    columns=["temp"],
    title="Outdoor Temperature — Horw (PLZ 6048)",
    ylab="Temperature [°C]",
)
fig.show()

In [ ]:
fig = plot_timeseries(
    df_plz,
    columns=["globrad"],
    title="Global Radiation — Horw (PLZ 6048)",
    ylab="Radiation [W/m²]",
)
fig.show()

## Available Sensors

Both providers support the same four sensor types:

| Sensor key | Description | Unit |
|---|---|---|
| `"temp"` | Air temperature | °C |
| `"globrad"` | Global radiation | W/m² |
| `"relhum"` | Relative humidity | % |
| `"rain"` | Precipitation | mm |

:::{admonition} Which provider should I use?
:class: tip dropdown

| Criteria | MeteoSwiss | Agrometeo |
|---|---|---|
| **Data download** | Not directly (use IDAWEB portal) | Yes, via API |
| **Station density** | ~160 stations | ~217 stations |
| **Altitude matching** | Yes (±150 m filter) | No |
| **Best for** | Finding the official reference station | Quick data access in code |

**Recommendation:** Use **Agrometeo** when you need data directly in Python. Use **MeteoSwiss** when you need the official reference station ID (e.g. for IDAWEB or regulatory purposes).
:::

## Summary

| Task | MeteoSwiss | Agrometeo |
|---|---|---|
| Find nearest station | `meteo_swiss.find_nearest_station(lat, lon, alt, sensor)` | `agroweather.find_nearest_station(lat, lon, sensor)` |
| List all stations | `meteo_swiss.get_current_station_data()` | `agroweather.get_station_data()` |
| Download data by station | — | `agroweather.download_data(station_id, start, end)` |
| Download data by PLZ | — | `agroweather.download_data_by_plz(plz, start, end)` |
| PLZ → coordinates | `geopy.get_coordindates_ch_plz(plz)` | (built into `download_data_by_plz`) |
| Get altitude | `geopy.get_altitude_lat_long(lat, lon)` | — |